In [9]:
!pip install -q -U pip
!pip install -q transformers datasets accelerate evaluate scikit-learn optuna sentencepiece
!pip install -q wandb

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import pandas as pd
train = pd.read_csv('dataset/train.csv', delimiter='|', on_bad_lines='skip')
val = pd.read_csv('dataset/val.csv', delimiter='|', on_bad_lines='skip')
test = pd.read_csv('dataset/test.csv', delimiter='|', on_bad_lines='skip')
print('Train', train.shape, 'Val', val.shape, 'Test', test.shape)
print('Train label dist:\n', train['Label_Sentimen'].value_counts(normalize=True))

Train (2994, 2) Val (374, 2) Test (375, 2)
Train label dist:
 Label_Sentimen
Positif    0.513360
Negatif    0.384102
Netral     0.102538
Name: proportion, dtype: float64


The `ParserError` indicates that `pd.read_csv` is struggling to correctly interpret the file's structure. This often happens if the delimiter is not what pandas expects, or if there are inconsistencies in the file. Let's inspect the first few lines of `train.csv` directly to understand its actual format and determine the correct delimiter.

In [19]:
from datasets import load_dataset, Value
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_files = {
    "train": "dataset/train.csv",
    "validation": "dataset/val.csv",
    "test": "dataset/test.csv"
}
raw = load_dataset("csv", data_files=data_files, delimiter='|', on_bad_lines='skip')

# Standardize column names
text_col = "Teks_Gabungan"
label_col = "Label_Sentimen"

# 1. Convert string labels to ClassLabel (integers) using the built-in method
# This ensures the 'labels' are flat integers and not lists.
raw = raw.class_encode_column(label_col)
raw = raw.rename_column(label_col, "labels")

def preprocess(examples):
    # Tokenize the texts
    result = tokenizer(examples[text_col], truncation=True, padding="max_length", max_length=512)
    # labels is already an integer from class_encode_column
    result["labels"] = examples["labels"]
    return result

# 2. Map preprocessing and remove unnecessary columns
tokenized = raw.map(
    preprocess,
    batched=True,
    remove_columns=raw["train"].column_names
)

tokenized.set_format(type="torch")
print(tokenized["train"].features)
display(tokenized)

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

{'labels': ClassLabel(names=['Negatif', 'Netral', 'Positif']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2994
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 374
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 375
    })
})

In [20]:
# training setup
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# 1. Tambah Dropout untuk mencegah overfitting
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.2,      # Naikkan dropout
    attention_probs_dropout_prob=0.2
)

# 2. Freeze beberapa layer awal BERT (Hanya latih 6 layer terakhir + head)
# Ini sangat ampuh untuk mencegah overfitting pada dataset kecil
for name, param in model.named_parameters():
    if 'bert.encoder.layer' in name:
        layer_num = int(name.split('.')[3])
        if layer_num < 6: # Freeze layer 0 sampai 5
            param.requires_grad = False
    elif 'bert.embeddings' in name:
        param.requires_grad = False

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/finpro_/checkpoints/indobert-finetune",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.05,            # Naikkan weight decay untuk regularisasi ekstra
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    save_total_limit=3
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# 3. Class Weights (Wajib karena data imbalanced)
labels_array = np.array(tokenized["train"]["labels"])
class_wts = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
weights_tensor = torch.tensor(class_wts, dtype=torch.float)

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Pindahkan bobot ke GPU (jika menggunakan GPU)
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
# Cell 6 — train
trainer.train()
# save final model + tokenizer
trainer.save_model("/content/drive/MyDrive/finpro_/final_models/indobert-finetuned")
tokenizer.save_pretrained("/content/drive/MyDrive/finpro_/final_models/indobert-finetuned")

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,8.556688,0.937731,0.628342,0.534284,0.532072,0.528569
2,6.946842,0.858697,0.719251,0.656062,0.600359,0.607573
3,6.399431,0.794648,0.745989,0.630859,0.600755,0.604605
4,5.393416,0.842904,0.751337,0.649758,0.585526,0.591217
5,5.095146,0.729231,0.719251,0.628753,0.650859,0.630607
6,4.671605,0.775858,0.745989,0.646138,0.647021,0.641872
7,4.232426,0.762392,0.772727,0.672642,0.670352,0.671313
8,3.851804,0.762596,0.772727,0.674558,0.672088,0.673289
9,3.611980,0.769711,0.772727,0.679061,0.680860,0.679781
10,3.486030,0.773392,0.770053,0.673434,0.679703,0.676088


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/finpro_/final_models/indobert-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/finpro_/final_models/indobert-finetuned/tokenizer.json')

In [ ]:
import json
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

metrics = trainer.evaluate(tokenized["test"])
print(metrics)

# Predict and confusion matrix
preds_output = trainer.predict(tokenized["test"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1,2], yticklabels=[0,1,2])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix (test)')
plt.show()

# Save metrics + cm
out_dir = "/content/drive/MyDrive/finpro_/final_models/indobert-finetuned"
with open(out_dir + "/metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
np.savetxt(out_dir + "/confusion_matrix.csv", cm, delimiter=",", fmt='%d')

In [ ]:
!zip -r /content/drive/MyDrive/finpro/final_models/indobert-finetuned.zip /content/drive/MyDrive/finpro/final_models/indobert-finetuned